In [ ]:
%pip install matplotlib
%pip install seaborn
%pip install torch torchvision torchaudio

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import itertools

# Configuração do dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")


# Estatísticas descritiva das variáveis usando df.describe()
df = pd.read_csv('../data/breast_cancer.csv')
df.describe()

# Histogramas individuais de cada característica

df.drop(columns=['id'], errors='ignore').hist(bins=30, figsize=(20, 20))
plt.suptitle("Histogramas das variáveis", fontsize=16)
plt.tight_layout()
plt.show()

# Boxplots por classe (análise de dispersão, outliers e comparação)

print(df.columns)


# Exclude columns with all NaN values
valid_columns = df.drop(columns=['id', 'diagnosis'], errors='ignore').dropna(axis=1, how='all').columns

for col in valid_columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x='diagnosis', y=col, data=df)
    plt.title(f'{col} por classe (diagnosis)')
    plt.tight_layout()
    plt.show()

# Removendo colunas não numéricas e a 'id' (se existir)
numerical_df = df.select_dtypes(include=['float64', 'int64']).drop(columns=['id'], errors='ignore')

plt.figure(figsize=(16, 12))
sns.heatmap(numerical_df.corr(), cmap='coolwarm', annot=False)
plt.title("Mapa de calor da correlação entre variáveis numéricas")
plt.show()


# Lista de pares com correlação forte (> 0.9 e < 1.0)
corr_matrix = numerical_df.corr().abs()  
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.9:
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            corr_value = corr_matrix.iloc[i, j]
            high_corr_pairs.append((col1, col2, corr_value))

high_corr_pairs = sorted(high_corr_pairs, key=lambda x: -x[2])

for pair in high_corr_pairs:
    print(f"{pair[0]} e {pair[1]} → correlação = {pair[2]:.4f}")


# Verificação da distribuição das classes alvo

sns.countplot(x='diagnosis', data=df)
plt.title("Distribuição da variável alvo (diagnosis)")
plt.show()

# Percentual
print(df['diagnosis'].value_counts(normalize=True) * 100)



# Discussão sobre possíveis variáveis redundantes ou fortemente correlacionadas.

# Com base na matriz de correlação, observou-se que algumas variáveis apresentam alta correlação linear entre si (coeficientes superiores a 0.9). Exemplos notáveis incluem:

# radius_mean com perimeter_mean e area_mean;

# perimeter_worst com radius_worst e area_worst;

# concavity_mean com concave points_mean.

# Essas correlações indicam que essas variáveis carregam informações similares e, portanto, são candidatas à remoção ou redução de dimensionalidade (ex: PCA) para evitar redundância e multicolinearidade em etapas posteriores.

# Além disso, a análise do Fator de Inflação da Variância (VIF) reforçou essa conclusão. Variáveis como:

# radius_mean,

# perimeter_mean,

# area_mean,

# radius_worst,

# apresentaram VIFs significativamente elevados (acima de 10), confirmando a presença de multicolinearidade severa.

# Portanto, é recomendado avaliar a eliminação ou agregação de variáveis altamente correlacionadas antes da modelagem preditiva.

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Python310\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Python310\python.exe -m pip install --upgrade pip' command.
